# Case Técnico PySpark
Análise de dados de e-commerce com foco em qualidade, agregações por cliente e métricas estatísticas.

In [65]:
## Lista de importações
import pandas as pd
import matplotlib as plt
import os
import warnings
import logging
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructType, StructField
from pyspark.storagelevel import StorageLevel
from functools import reduce
from pyspark.sql.functions import broadcast

# Suprime avisos não críticos
warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)

# Define variáveis globais de caminho
CWD = os.getcwd()
CLIENTES_PATH = os.path.join(CWD, 'data/clients/data.json')
PEDIDOS_PATH = os.path.join(CWD, 'data/pedidos/data.json')

# Inicializa sessão PySpark
spark = (
    SparkSession.builder
    .appName("CaseTecnico")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

# Ajusta nível de log após criar a sessão
spark.sparkContext.setLogLevel("ERROR")



In [66]:
# Esquemas explícitos evitam passagem extra de inferência
CLIENTES_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

PEDIDOS_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5, 2), True),
])


def load_data_from_json(path: str, schema: StructType, min_partitions: int | None = None) -> DataFrame:
    """Carrega JSONL com schema explícito, de forma lazy e com reparticionamento opcional. Persiste o DataFrame em memória/disco."""
    df = (
        spark.read
        .schema(schema)
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json(path)
    )

    if min_partitions is not None and df.rdd.getNumPartitions() < min_partitions:
        df = df.repartition(min_partitions)

    df = df.persist(StorageLevel.MEMORY_AND_DISK)
    return df


clientes_df = load_data_from_json(CLIENTES_PATH, CLIENTES_SCHEMA)
pedidos_df= load_data_from_json(PEDIDOS_PATH, PEDIDOS_SCHEMA, min_partitions=32)

print("Partições de clientes:", clientes_df.rdd.getNumPartitions())
print("Partições de pedidos:", pedidos_df.rdd.getNumPartitions())
print("Esquemas carregados com sucesso")

Partições de clientes: 1
Partições de pedidos: 32
Esquemas carregados com sucesso


In [67]:
clientes_df.summary('count').show()
clientes_df.printSchema()

+-------+-----+-----+
|summary|   id| name|
+-------+-----+-----+
|  count|10001|10001|
+-------+-----+-----+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [68]:
pedidos_df.summary().show()
pedidos_df.printSchema()

repeticoes_client_id_df = (
    pedidos_df
    .groupBy("client_id")
    .count()
    .orderBy(F.col("count").desc())
)

repeticoes_client_id_df.show(20, truncate=False)

+-------+--------------------+-----------------+-----------------+
|summary|                  id|        client_id|            value|
+-------+--------------------+-----------------+-----------------+
|  count|             1100000|          1100000|          1050000|
|   mean| 5.000917537925636E7|64178.34457090909|        45.670702|
| stddev|2.8861777376239363E7|59263.14505173698|35.76652492877208|
|    min|                  75|                0|           -99.99|
|    25%|            25050740|             4992|            21.97|
|    50%|            50001242|             9989|            47.99|
|    75%|            75013737|           123456|             74.0|
|    max|            99999981|           123456|            99.99|
+-------+--------------------+-----------------+-----------------+

root
 |-- id: long (nullable = true)
 |-- client_id: long (nullable = true)
 |-- value: decimal(5,2) (nullable = true)

+---------+------+
|client_id|count |
+---------+------+
|123456   |549539|

## Relatório de Qualidade dos Dados
Identificação de pedidos com falhas de qualidade e consolidação dos motivos por pedido.

In [ ]:
def regra_falha(df, motivo: str, ordem: int):
    return df.select(
        F.col("id"),
        F.lit(motivo).alias("motivo"),
        F.lit(ordem).alias("ordem_regra")
    )

# 1) ID de pedido duplicado
ids_duplicados_df = (
    pedidos_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .select("id")
)
ids_duplicados = regra_falha(ids_duplicados_df, "id_duplicado", 1)

# 2) Pedido sem valor
sem_valor = regra_falha(
    pedidos_df.filter(F.col("value").isNull()),
    "pedido_sem_valor",
    2
)

# 3) Pedido com cliente inexistente (somente client_id preenchido)
cliente_inexistente = (
    pedidos_df.alias("p")
    .filter(
        F.col("p.client_id").isNotNull() &
        (F.col("p.client_id") > 0)
    )
    .join(
        broadcast(clientes_df.select(F.col("id").alias("client_id_ref")).alias("c")),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left_anti"
    )
    .select(F.col("p.id").alias("id"))
    .transform(lambda df: regra_falha(df, "cliente_inexistente", 3))
)

# 4) ID nulo
id_nulo = regra_falha(
    pedidos_df.filter(F.col("id").isNull()),
    "id_nulo",
    4
)

# 5) client_id nulo
client_id_nulo = regra_falha(
    pedidos_df.filter(F.col("client_id").isNull()),
    "client_id_nulo",
    5   
)

# 6) ID inválido (< 0)
id_invalido = regra_falha(
    pedidos_df.filter(F.col("id").isNotNull() & (F.col("id") < 0)),
    "id_invalido_menor_igual_zero",
    6
)

# 7) client_id inválido (<= 0)
client_id_invalido = regra_falha(
    pedidos_df.filter(F.col("client_id").isNotNull() & (F.col("client_id") < 0)),
    "client_id_invalido_menor_igual_zero",
    7
)

# 8) Valor zero
valor_zero = regra_falha(
    pedidos_df.filter(F.col("value") == 0),
    "pedido_com_valor_zero",
    8
)

# 9) Retorno inválido: verifica se pedidos com o mesmo ID têm valor positivo suficiente
# Regra: se um ID tem valor negativo, deve existir valor positivo no mesmo ID >= |valor_negativo|
# Agrupa por ID e calcula total positivo e negativo
pedidos_por_id_df = (
    pedidos_df
    .filter(F.col("value").isNotNull())
    .groupBy("id")
    .agg(
        F.sum(F.when(F.col("value") > 0, F.col("value")).otherwise(0)).alias("total_positivo"),
        F.abs(F.sum(F.when(F.col("value") < 0, F.col("value")).otherwise(0))).alias("total_negativo_abs")
    )
)

# IDs com valores negativos sem valor positivo suficiente para cobrir
retornos_invalidos_df = (
    pedidos_por_id_df
    .filter(
        (F.col("total_negativo_abs") > 0) &  # Há valor negativo
        (F.col("total_positivo") < F.col("total_negativo_abs"))  # Positivo não cobre o negativo
    )
    .select("id")
)

retornos_invalidos = regra_falha(retornos_invalidos_df, "retorno_sem_cobertura_positiva", 9)

#10 valor pertence a anomalia de repetição de client_id, onde um mesmo client_id tem mais de da metade de todos os pedidos, o que é improvável e pode indicar erro de cadastro ou fraude. Para identificar isso, podemos calcular a frequência de cada client_id e marcar aqueles que ultrapassarem esse limite.

anomalia_repeticao_client_id_df = regra_falha(
        pedidos_df.filter((F.col("client_id").isNotNull()) & (F.col("client_id") == 123456)),
        "anomalia_repeticao_client_id",
        10
    )

# União de todas as regras (base normalizada de eventos de falha)
regras_falhas = [
    sem_valor, ids_duplicados, cliente_inexistente, id_nulo, client_id_nulo,
    id_invalido, client_id_invalido, valor_zero, retornos_invalidos, anomalia_repeticao_client_id_df
]

falhas_eventos_df = (
    reduce(lambda acc, d: acc.unionByName(d), regras_falhas)
    .dropDuplicates(["id", "motivo"])
)

# Consolidado por pedido: concatenação estável e ordenada por prioridade da regra
falhas_df = (
    falhas_eventos_df
    .groupBy("id")
    .agg(
        F.sort_array(F.collect_set(F.struct("ordem_regra", "motivo"))).alias("motivos_ordenados"),
        F.min("ordem_regra").alias("min_ordem")
    )
    .select(
        "id",
        F.concat_ws(", ", F.expr("transform(motivos_ordenados, x -> x.motivo)")).alias("motivo"),
        "min_ordem"
    )
    .orderBy("min_ordem", "id")
    .select("id", "motivo")
)

# Contagem correta por categoria (motivo individual)
erros_por_categoria_df = (
    falhas_eventos_df
    .groupBy("motivo")
    .agg(F.count("*").alias("qtd_erros"))
    .orderBy(F.col("qtd_erros").desc(), F.col("motivo").asc())
)

falhas_df.show(100, truncate=False)
erros_por_categoria_df.show(truncate=False)
falhas_df.agg(F.count("*").alias("total_erros")).show()



+------+--------------------------------------------------------------------------------------------+
|id    |motivo                                                                                      |
+------+--------------------------------------------------------------------------------------------+
|1534  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva, Anomalia_repeticao_client_id|
|3443  |id_duplicado, Anomalia_repeticao_client_id                                                  |
|3502  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva, Anomalia_repeticao_client_id|
|3679  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva, Anomalia_repeticao_client_id|
|4322  |id_duplicado, pedido_sem_valor, Anomalia_repeticao_client_id                                |
|4724  |id_duplicado, pedido_sem_valor, Anomalia_repeticao_client_id                                |
|5774  |id_duplicado, pedido_sem_valor, Anomalia_repeticao_client_id              

In [51]:
# 1) Filtra linhas com valores válidos
pedidos_with_valid_values = (
    pedidos_df
    .select("id", "client_id", "value")
    .filter(
        F.col("value").isNotNull() 
        & (F.col("value") > 0)
        & F.col("id").isNotNull() 
        & (F.col("id") >= 0)
        & F.col("client_id").isNotNull() 
        & (F.col("client_id") >= 0)
        & (F.col("client_id") != 123456)  # Exclui client_id com anomalia de repetição
    )
)

# 2) Verifica duplicidades somente entre os registros válidos
# (se um ID aparece múltiplas vezes, mas só uma é válida, ele é mantido)
ids_duplicados_validos_df = (
    pedidos_with_valid_values
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Prepara IDs de clientes como DataFrame (evita lista Python)
clientes_ids_df = clientes_df.select(F.col("id").alias("client_id_ref")).distinct()

# 3) Mantém somente pedidos válidos não duplicados e com cliente existente
pedidos_validos_df = (
    pedidos_with_valid_values
    # Anti-join para excluir IDs duplicados entre valores válidos
    .join(broadcast(ids_duplicados_validos_df), F.col("id") == F.col("dup_id"), "left_anti")
    # Inner join para validar existência do cliente
    .join(broadcast(clientes_ids_df), F.col("client_id") == F.col("client_id_ref"), "inner")
    .select("id", "client_id", "value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos = pedidos_validos_df.count()
total_pedidos = pedidos_df.count()

print("Total de pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)

Total de pedidos: 1100000
Pedidos válidos: 498065


In [64]:
# 2. Agregação por cliente
cliente_totals_df = (
    pedidos_validos_df
    .groupBy("client_id")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum("value").cast(DecimalType(11, 2)).alias("valor_total"),
    )
    .join(
        broadcast(clientes_df.select(
            F.col("id").alias("client_id_ref"), 
            F.col("name").alias("client_name")
        )),
        F.col("client_id") == F.col("client_id_ref"),
        "inner"
    )
    .select(
        F.col("client_id").alias("id_cliente"),
        F.col("client_name").alias("nome_cliente"),
        F.col("qtd_pedidos"),
        F.col("valor_total")
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

cliente_totals_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|9047      |Zachary Reis      |64         |4213.80    |
|4494      |Vitor Marques     |72         |4185.48    |
|2379      |Gustavo Pontes    |71         |4022.72    |
|2756      |Inês Siqueira     |65         |3977.05    |
|7543      |Vitória Andrade   |76         |3959.06    |
|8317      |Sofia Castro      |72         |3954.30    |
|8566      |Tereza Leal       |72         |3940.74    |
|9147      |Zachary Reis      |64         |3932.56    |
|849       |Breno Soares      |72         |3932.45    |
|5221      |Yasmin Carvalho   |70         |3922.36    |
|6135      |Mariana Melo      |75         |3899.41    |
|1704      |Eduardo Santos    |61         |3871.00    |
|1942      |Ulisses Moraes    |68         |3836.04    |
|9266      |Tereza Leal       |70         |3826.47    |
|2695      |Wanda Silva       |63         |3824.

In [53]:
# 3. Métricas estatísticas (média, mediana, P10 e P90)
stats = cliente_totals_df.agg(F.mean("valor_total").alias("media")).collect()[0]
media = stats["media"]

# Calcula os quantis em uma única passagem
percentil_10, mediana, percentil_90 = cliente_totals_df.approxQuantile("valor_total", [0.1, 0.5, 0.9], 0.01)

print(f"Valor médio total por cliente: {media:.2f}")
print(f"Mediana do valor total por cliente: {mediana:.2f}")
print(f"10º percentil do valor total por cliente: {percentil_10:.2f}")
print(f"90º percentil do valor total por cliente: {percentil_90:.2f}")

Valor médio total por cliente: 2514.53
Mediana do valor total por cliente: 2497.36
10º percentil do valor total por cliente: 1989.81
90º percentil do valor total por cliente: 3036.40


In [54]:
# 4. Clientes com valor total acima da média
clientes_acima_media_df = (
    cliente_totals_df
    .filter(F.col("valor_total") > media)
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|9047      |Zachary Reis      |64         |4213.80    |
|4494      |Vitor Marques     |72         |4185.48    |
|2379      |Gustavo Pontes    |71         |4022.72    |
|2756      |Inês Siqueira     |65         |3977.05    |
|7543      |Vitória Andrade   |76         |3959.06    |
|8317      |Sofia Castro      |72         |3954.30    |
|8566      |Tereza Leal       |72         |3940.74    |
|9147      |Zachary Reis      |64         |3932.56    |
|849       |Breno Soares      |72         |3932.45    |
|5221      |Yasmin Carvalho   |70         |3922.36    |
|6135      |Mariana Melo      |75         |3899.41    |
|1704      |Eduardo Santos    |61         |3871.00    |
|1942      |Ulisses Moraes    |68         |3836.04    |
|9266      |Tereza Leal       |70         |3826.47    |
|2695      |Wanda Silva       |63         |3824.

In [55]:
# 5. Clientes dentro da média truncada (entre P10 e P90)
clientes_media_truncada_df = (
    cliente_totals_df
    .filter(
        (F.col("valor_total") >= percentil_10) &
        (F.col("valor_total") <= percentil_90)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+----------+-----------------+-----------+-----------+
|id_cliente|nome_cliente     |qtd_pedidos|valor_total|
+----------+-----------------+-----------+-----------+
|20        |Wagner Teixeira  |61         |3036.40    |
|4449      |Breno Soares     |55         |3036.36    |
|7961      |Orlando Guedes   |59         |3036.09    |
|5728      |Felipe Cunha     |62         |3036.06    |
|6575      |Caio Teles       |51         |3035.87    |
|4929      |Giovanna Ramos   |57         |3035.61    |
|2975      |Caio Teles       |53         |3035.46    |
|6851      |Diogo Tavares    |53         |3035.37    |
|4665      |Sérgio Esteves   |53         |3035.15    |
|3822      |Zeca Mendes      |58         |3034.88    |
|7214      |Otávio Cardoso   |53         |3034.62    |
|5526      |Daniel Moreira   |63         |3034.27    |
|8313      |Natália Ribeiro  |63         |3034.13    |
|1139      |Rafaela Vieira   |53         |3034.06    |
|8579      |Gustavo Pontes   |60         |3033.96    |
|9561     

## Análise Complementar: Outlier (Cliente 123456)
### Justificativa técnica para exclusão em análises estatísticas
**1. Anomalia estatística extrema**
- Cliente 123456: **494.418 pedidos** | Valor total: **R$ 24.946.507**
- Demais clientes: ~60 a 75 pedidos | Valor total: **R$ 3.000 a R$ 4.200**
- Diferença aproximada: **6.500x** em quantidade de pedidos

**2. Distribuição de valores suspeita**
- Muitos pedidos com valores repetidos (ex.: 99,72; 59,25; 51,41)
- Repetição em alta frequência é improvável em dados reais

**3. Impacto nos indicadores estatísticos**
- **Com o cliente 123456:** média inflacionada
- **Sem o cliente 123456:** média normalizada, mediana praticamente estável

**4. Conclusão**
O cliente 123456 é tratado como provável erro de dados (duplicação, massa de teste ou corrupção), devendo ser excluído de análises estatísticas para preservar a representatividade dos resultados.

In [ ]:
# 1) Filtra linhas com valores válidos
pedidos_with_valid_values_b = (
    pedidos_df
    .select("id", "client_id", "value")
    .filter(
        F.col("value").isNotNull() 
        & (F.col("value") > 0)
        & F.col("id").isNotNull() 
        & (F.col("id") >= 0)
        & F.col("client_id").isNotNull() 
        & (F.col("client_id") >= 0)
        #& (F.col("client_id") != 123456)  # Exclui client_id com anomalia de repetição
    )
)

# 2) Verifica duplicidades somente entre os registros válidos
# (se um ID aparece múltiplas vezes, mas só uma é válida, ele é mantido)
ids_duplicados_validos_df_b = (
    pedidos_with_valid_values_b
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Prepara IDs de clientes como DataFrame (evita lista Python)
clientes_ids_df = clientes_df.select(F.col("id").alias("client_id_ref")).distinct()

# 3) Mantém somente pedidos válidos não duplicados e com cliente existente
pedidos_validos_df_b = (
    pedidos_with_valid_values_b
    # Anti-join para excluir IDs duplicados entre valores válidos
    .join(broadcast(ids_duplicados_validos_df_b), F.col("id") == F.col("dup_id"), "left_anti")
    # Inner join para validar existência do cliente
    .join(broadcast(clientes_ids_df), F.col("client_id") == F.col("client_id_ref"), "inner")
    .select("id", "client_id", "value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos = pedidos_validos_df_b.count()
total_pedidos = pedidos_df.count()

print("Total de pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)

Total de pedidos: 1100000
Pedidos válidos: 989978


In [ ]:
pedidos_ines = (
    pedidos_validos_df_b.alias("pedidos")
    .filter(F.col("client_id") == 123456)
    .orderBy(F.col("value").desc())
    .select("id", "value")
)

ines_valores_repetidos_df = (
    pedidos_ines.groupBy("value")
    .agg(F.count("*").alias("qtd_repeticoes"))
    .filter(F.col("qtd_repeticoes") > 1)
    .orderBy(F.col("qtd_repeticoes").desc(), F.col("value").asc())
)

ines_valores_repetidos_df.show(10, truncate=False)

+-----+--------------+
|value|qtd_repeticoes|
+-----+--------------+
|99.72|79            |
|59.25|77            |
|51.41|76            |
|75.98|75            |
|58.09|74            |
|71.72|74            |
|83.85|74            |
|84.41|74            |
|6.86 |73            |
|31.87|73            |
+-----+--------------+
only showing top 10 rows


In [ ]:
pedidos_validos_sem = pedidos_validos_df.count()
pedidos_validos_com = pedidos_validos_df_b.count()


print("=" * 60)
print("ANÁLISE COM DADOS LIMPOS (sem cliente outlier 123456)")
print("=" * 60)
print(f"Pedidos válidos (sem 123456): {pedidos_validos_sem}")
print(f"Pedidos válidos (com 123456): {pedidos_validos_com}")
print(f"Diferença: {pedidos_validos_com - pedidos_validos_sem}")
print("=" * 60)

ANÁLISE COM DADOS LIMPOS (sem cliente outlier 123456)
Pedidos válidos (sem 123456): 498065
Pedidos válidos (com 123456): 989978
Diferença: 491913


In [ ]:

def calcular_metricas(df_pedidos):
    cliente_totais_df = (
        df_pedidos
        .groupBy("client_id")
        .agg(F.sum("value").cast(DecimalType(11, 2)).alias("valor_total"))
    )

    media_local = cliente_totais_df.agg(F.mean("valor_total").alias("media")).first()["media"]
    p10_local, mediana_local, p90_local = cliente_totais_df.approxQuantile(
        "valor_total", [0.1, 0.5, 0.9], 0.01
    )
    return float(media_local), p10_local, mediana_local, p90_local


# Com outlier (base original válida)
media_com, p10_com, mediana_com, p90_com = calcular_metricas(pedidos_validos_df_b)

# Sem outlier (base limpa)
media_sem, p10_sem, mediana_sem, p90_sem = calcular_metricas(pedidos_validos_df)

print("=== COM outlier (client_id = 123456) ===")
print(f"Média:    {media_com:.2f}")
print(f"Mediana:  {mediana_com:.2f}")
print(f"P10:      {p10_com:.2f}")
print(f"P90:      {p90_com:.2f}")

print("\n=== SEM outlier (client_id = 123456) ===")
print(f"Média:    {media_sem:.2f}")
print(f"Mediana:  {mediana_sem:.2f}")
print(f"P10:      {p10_sem:.2f}")
print(f"P90:      {p90_sem:.2f}")

print("\n=== Diferença (COM - SEM) ===")
print(f"Δ Média:   {(media_com - media_sem):.2f}")
print(f"Δ Mediana: {(mediana_com - mediana_sem):.2f}")
print(f"Δ P10:     {(p10_com - p10_sem):.2f}")
print(f"Δ P90:     {(p90_com - p90_sem):.2f}")

=== COM outlier (client_id = 123456) ===
Média:    4996.23
Mediana:  2485.49
P10:      2003.43
P90:      3007.03

=== SEM outlier (client_id = 123456) ===
Média:    2514.53
Mediana:  2496.55
P10:      2011.79
P90:      3017.93

=== Diferença (COM - SEM) ===
Δ Média:   2481.70
Δ Mediana: -11.06
Δ P10:     -8.36
Δ P90:     -10.90
